# Práctica: destilar un modelo profesor en un modelo estudiante

[![Abrir en Colab](https://colab.research.google.com/assets/colab-badge.svg)](https://colab.research.google.com/github/mosqueteroweb/IAconceptos/blob/main/notebooks/destilacion.ipynb)

Un modelo **profesor** (DistilBERT, 67 millones de parámetros), ya entrenado para saber si una reseña de cine en
inglés es positiva o negativa, va a enseñar a un modelo **estudiante** 15 veces más pequeño (BERT-tiny,
4,4 millones). La pregunta de la práctica: ¿puede el profesor sustituir a las personas que etiquetan datos?


> **Cómo usar este cuaderno.** No hace falta saber programar. Ejecuta las celdas **en orden**, de arriba abajo,
> con el botón ▶ que aparece a su izquierda (o con **Mayús + Intro**). Lee el texto que hay antes de cada celda:
> explica qué va a pasar. Si algo falla, usa *Entorno de ejecución → Reiniciar y ejecutar todo*.


**Antes de empezar:** en Colab, *Entorno de ejecución → Cambiar tipo → GPU T4*. Tarda unos 10 minutos.

Datos: [SST-2](https://huggingface.co/datasets/stanfordnlp/sst2), reseñas de cine en inglés etiquetadas por personas.
Material del curso [IAconceptos](https://mosqueteroweb.github.io/IAconceptos/) · MosqueteroWeb · CC BY-SA 4.0.

In [ ]:
!pip install -q transformers datasets scikit-learn accelerate

## Paso 1. El profesor, el estudiante y los datos

- **Profesor:** `distilbert-base-uncased-finetuned-sst-2-english`, ya especializado en sentimientos.
- **Estudiante:** `google/bert_uncased_L-2_H-128_A-2` (conocido como *BERT-tiny*), un BERT minúsculo que todavía no
  sabe nada de sentimientos.
- **Datos:** 30 000 frases de entrenamiento y las 872 de validación de SST-2.

Los dos modelos comparten el mismo vocabulario (el de BERT en minúsculas), así que leen exactamente los mismos tokens.

In [ ]:
import torch, time
from datasets import load_dataset
from transformers import AutoTokenizer, AutoModelForSequenceClassification

torch.manual_seed(0)
PROFESOR = "distilbert-base-uncased-finetuned-sst-2-english"
ESTUDIANTE = "google/bert_uncased_L-2_H-128_A-2"  # «BERT-tiny»: 2 capas, 128 dimensiones
dispositivo = "cuda" if torch.cuda.is_available() else ("mps" if torch.backends.mps.is_available() else "cpu")

tokenizador = AutoTokenizer.from_pretrained(PROFESOR)
profesor = AutoModelForSequenceClassification.from_pretrained(PROFESOR).to(dispositivo).eval()

datos = load_dataset("stanfordnlp/sst2")
mezclado = datos["train"].shuffle(seed=0)

def tokenizar(lote):
    return tokenizador(lote["sentence"], truncation=True, max_length=64)
pocas = mezclado.select(range(1_000)).map(tokenizar, batched=True).rename_column("label", "labels")
muchas = mezclado.select(range(30_000)).map(tokenizar, batched=True).rename_column("label", "labels")
validacion = datos["validation"].map(tokenizar, batched=True).rename_column("label", "labels")

millones = lambda m: sum(p.numel() for p in m.parameters()) / 1e6
print(f"Profesor: {millones(profesor):.1f} M parámetros")
print(f"Estudiante: {millones(AutoModelForSequenceClassification.from_pretrained(ESTUDIANTE, num_labels=2)):.1f} M parámetros")

## Paso 2. Lo que sabe el profesor: «etiquetas blandas»

El profesor no solo dice «positiva»: da una **probabilidad** para cada opción. Esa información extra —que una frase
es *casi* positiva, o que duda— es el «conocimiento oscuro» que el estudiante intentará copiar. Fíjate también en que
el profesor **no es perfecto**.

In [ ]:
frases = ["A masterpiece, moving and beautifully acted.",
          "It was fine, I guess, but I wouldn't watch it again.",
          "Two hours of my life I will never get back."]
with torch.no_grad():
    entradas = tokenizador(frases, return_tensors="pt", padding=True).to(dispositivo)
    probabilidades = torch.softmax(profesor(**entradas).logits, dim=-1)
for frase, (neg, pos) in zip(frases, probabilidades.tolist()):
    print(f"{pos:6.1%} positiva · {neg:6.1%} negativa  ←  {frase}")

## Paso 3. El entrenador de destilación

El estudiante destilado **no ve las etiquetas humanas**. Su único objetivo es que sus probabilidades se parezcan a las
del profesor. Las dos distribuciones se «suavizan» con una **temperatura** (2) para que se vean mejor los matices, y la
diferencia entre ellas se mide con la *divergencia de Kullback-Leibler*.

In [ ]:
import torch.nn.functional as F
from transformers import Trainer, TrainingArguments, DataCollatorWithPadding
from sklearn.metrics import accuracy_score

TEMPERATURA = 2.0

class EntrenadorDestilacion(Trainer):
    def compute_loss(self, model, inputs, return_outputs=False, num_items_in_batch=None):
        inputs.pop("labels")                      # ¡las etiquetas humanas se tiran!
        salida = model(**inputs)
        profesor.to(model.device)
        with torch.no_grad():
            logits_profesor = profesor(**inputs).logits
        perdida = F.kl_div(F.log_softmax(salida.logits / TEMPERATURA, dim=-1),
                           F.softmax(logits_profesor / TEMPERATURA, dim=-1),
                           reduction="batchmean") * TEMPERATURA ** 2
        return (perdida, salida) if return_outputs else perdida

def metricas(prediccion):
    return {"accuracy": accuracy_score(prediccion.label_ids, prediccion.predictions.argmax(-1))}

def entrenar(datos, clase=Trainer, epocas=3):
    modelo = AutoModelForSequenceClassification.from_pretrained(ESTUDIANTE, num_labels=2)
    argumentos = TrainingArguments(output_dir="salida", num_train_epochs=epocas, learning_rate=1e-3,
                                   per_device_train_batch_size=64, per_device_eval_batch_size=128,
                                   eval_strategy="no", save_strategy="no", report_to=[], logging_steps=100)
    entrenador = clase(model=modelo, args=argumentos, train_dataset=datos, eval_dataset=validacion,
                       data_collator=DataCollatorWithPadding(tokenizador), compute_metrics=metricas)
    entrenador.train()
    return modelo, entrenador.evaluate()["eval_accuracy"]

## Paso 4. Entrenar tres estudiantes

Entrenamos tres copias idénticas del estudiante, que solo se diferencian en de quién aprenden:

1. **1 000 frases etiquetadas por personas.** Lo que tendrías si etiquetar a mano fuera caro.
2. **30 000 frases etiquetadas por personas.** El caso ideal, y el más caro.
3. **Las mismas 30 000 frases, sin etiquetas humanas, aprendiendo solo del profesor.**

In [ ]:
_, acierto_pocas = entrenar(pocas, epocas=5)
_, acierto_muchas = entrenar(muchas)
estudiante, acierto_destilado = entrenar(muchas, clase=EntrenadorDestilacion)

## Paso 5. Resultados

In [ ]:
evaluador = Trainer(model=profesor, args=TrainingArguments(output_dir="salida", report_to=[], per_device_eval_batch_size=128),
                    data_collator=DataCollatorWithPadding(tokenizador), compute_metrics=metricas)
acierto_profesor = evaluador.evaluate(validacion)["eval_accuracy"]

def velocidad(modelo, repeticiones=20):
    modelo = modelo.to(dispositivo).eval()
    entradas = tokenizador(list(validacion["sentence"][:128]), return_tensors="pt", padding=True).to(dispositivo)
    with torch.no_grad():
        modelo(**entradas)
        inicio = time.time()
        for _ in range(repeticiones):
            modelo(**entradas)
    return 128 * repeticiones / (time.time() - inicio)

print(f"{'':42s}{'Acierto':>9s}{'Frases/s':>10s}")
print(f"{'Profesor (67 M)':42s}{acierto_profesor:9.1%}{velocidad(profesor):10.0f}")
print(f"{'Estudiante · 1 000 etiquetas humanas':42s}{acierto_pocas:9.1%}")
print(f"{'Estudiante · 30 000 etiquetas humanas':42s}{acierto_muchas:9.1%}")
print(f"{'Estudiante · 30 000 frases, solo profesor':42s}{acierto_destilado:9.1%}{velocidad(estudiante):10.0f}")

## ¿Qué ha pasado?

- El estudiante es **15 veces más pequeño** y decenas de veces **más rápido** que el profesor. Pierde acierto: un
  modelo tan diminuto no puede igualar al grande.
- El estudiante destilado **no ha visto ni una etiqueta humana** y, aun así, queda prácticamente empatado (a un
  punto, arriba o abajo según la ejecución) con el que aprendió de 30 000 etiquetas puestas a mano, y muy por encima
  del que solo tenía 1 000. Las probabilidades del profesor enseñan casi tanto como el trabajo de las personas.
- La consecuencia práctica: con un buen profesor puedes entrenar un modelo pequeño con **cualquier texto sin
  etiquetar**, que es barato y abundante.

Esto es, a escala de juguete, lo que hacen las empresas cuando publican versiones «mini» o «flash» de sus modelos,
o lo que hizo DeepSeek al destilar su modelo de razonamiento R1 en modelos Qwen y Llama más pequeños.

**Para experimentar:** cambia `TEMPERATURA` (1, 4, 8) y vuelve a ejecutar los pasos 3 a 5. ¿Mejora o empeora el
estudiante destilado?